# 01 — Exploratory Data Analysis : CIC-IDS2017

**Objectifs de ce notebook :**
1. Charger les fichiers CSV bruts du dataset CIC-IDS2017
2. Comprendre la structure : nombre de lignes, colonnes, features
3. Identifier les classes (Benign vs attaques) et leur distribution
4. Détecter les problèmes : valeurs manquantes, infinis, doublons
5. Première intuition des features les plus discriminantes

**Livrable attendu :** un résumé clair des forces et faiblesses du dataset, qui guidera le preprocessing en semaine 3.

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Config affichage
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

# Chemins
DATA_RAW = Path('../data/raw/cic-ids-2017')
print('Dossier données :', DATA_RAW.resolve())
print('Existe :', DATA_RAW.exists())

## 2. Inventaire des fichiers

CIC-IDS2017 est découpé en 8 fichiers correspondant à différents jours d'enregistrement :
- Lundi : trafic normal uniquement
- Mardi : Brute Force (FTP, SSH)
- Mercredi : DoS / Heartbleed
- Jeudi matin : Web Attacks
- Jeudi après-midi : Infiltration
- Vendredi matin : Botnet
- Vendredi après-midi 1 : PortScan
- Vendredi après-midi 2 : DDoS

In [ ]:
csv_files = sorted(DATA_RAW.glob('*.csv'))
for f in csv_files:
    size_mb = f.stat().st_size / 1024**2
    print(f'{f.name:60s} {size_mb:>8.1f} MB')

## 3. Chargement et concaténation

In [ ]:
dfs = []
for f in csv_files:
    df_tmp = pd.read_csv(f, low_memory=False)
    df_tmp['source_file'] = f.name
    print(f'{f.name:60s} -> {df_tmp.shape}')
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)
print(f'\nDataset complet : {df.shape}')

In [ ]:
# Nettoyer les noms de colonnes (espaces parasites)
df.columns = df.columns.str.strip()
print('Colonnes :')
for c in df.columns:
    print(' -', c)

## 4. Distribution des classes

In [ ]:
label_col = 'Label'
counts = df[label_col].value_counts()
print(counts)

plt.figure(figsize=(10, 5))
counts.plot(kind='barh', color='steelblue')
plt.xscale('log')
plt.title('Distribution des classes (echelle log)')
plt.xlabel('Nombre d\'echantillons')
plt.tight_layout()
plt.savefig('../results/figures/01_class_distribution.png', dpi=120)
plt.show()

## 5. Valeurs manquantes et infinies

**Attention** : CIC-IDS2017 contient des `inf` et `-inf` qui ne sont pas détectés par `isna()`. À traiter spécifiquement.

In [ ]:
# Valeurs manquantes
na_summary = df.isna().sum()
print('Valeurs manquantes par colonne :')
print(na_summary[na_summary > 0].sort_values(ascending=False))

# Valeurs infinies
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_summary = np.isinf(df[numeric_cols]).sum()
print('\nValeurs infinies par colonne :')
print(inf_summary[inf_summary > 0].sort_values(ascending=False))

## 6. Doublons

In [ ]:
n_dups = df.duplicated().sum()
print(f'Nombre de doublons : {n_dups:,} ({100*n_dups/len(df):.2f}%)')

## 7. Statistiques descriptives

In [ ]:
df.describe().T.head(20)

## 8. Conclusions et points à traiter en preprocessing

_À compléter au fil de l'exploration :_

- [ ] Nombre de classes : ...
- [ ] Déséquilibre fort entre classes : ...
- [ ] Présence de NaN dans : ...
- [ ] Présence d'infinis dans : ...
- [ ] Pourcentage de doublons : ...
- [ ] Features à supprimer (constantes, IDs, IPs) : ...
- [ ] Hypothèses sur les features les plus discriminantes : ...